# **Finetuning de LLMs abertas**

Uma equipe de análise de dados de uma empresa precisa consultar informações do banco de dados com frequência para gerar relatórios e insights. Porém, nem todos os analistas têm conhecimentos avançados em SQL, o que gera uma dependência dos desenvolvedores para escrever essas consultas.

Além disso, o banco de dados possui informações de clientes que são sigilosas e a empresa não gostaria de utilizar grandes modelos de empresas que poderiam coletar dados e vazar informações.

Nosso papel nesse projeto é realizar o fine-tuning de um modelo de LLM aberta que converta comandos em linguagem natural para SQL, permitindo que os analistas façam suas consultas localmente e obtenham as informações que precisam sem precisar de suporte contínuo dos desenvolvedores e ao mesmo tempo não compartilhem os dados com APIs externas.

# **Gerando respostas com uma LLM**

### **Carregando o modelo Llama**

Para que seja possível utilizar uma LLM localmente, precisamos carregar um modelo mais leve, caso contrário o computador não conseguirá processar os resultados.

O [Unsloth](https://unsloth.ai/) fornece LLMs de código aberto e opções quantizadas dos modelos que reduz a memória necessária para o carregamento e melhora a velocidade de processamento:

- [Modelos de código aberto](https://huggingface.co/unsloth)

Vamos instalar a biblioteca Unsloth e pré-requisitos para carregar um modelo Llama. Precisamos utilizar uma GPU para utilização da biblioteca, portanto vamos usar a GPU T4 do Google Colab.

In [1]:
!pip install unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.3/82.3 MB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 53.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 54.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 50.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 

- Link git Unsloth: `'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git'`



In [2]:
!pip install --upgrade --no-deps 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git'

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-_74e81m9/unsloth_8efa65e886584f46a5c326b68ca0609e
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-_74e81m9/unsloth_8efa65e886584f46a5c326b68ca0609e
  Resolved https://github.com/unslothai/unsloth.git to commit 41d2ff40a6223e4f969710b6de98a272f2d0720a
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [3]:
!pip install  --no-deps torch xformers trl peft accelerate bitsandbytes triton

In [4]:
from unsloth import FastLanguageModel
import torch

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


Vamos utilizar o modelo LLama 3.1 com 8 bilhões de parâmetros. É um modelo de código aberto, por conta disso não precisamos de acessar nenhuma API, nem pagar nenhum valor para utilizar:

- [Llama 3.1-8B Hugging Face](https://huggingface.co/unsloth/Meta-Llama-3.1-8B)

In [5]:
checkpoint_modelo = 'unsloth/Meta-Llama-3.1-8B'

No momento de fazer o carregamento do modelo, vamos utilizar parâmetros para utilizar menos memória.

- dtype: None para detecção automática, Float16 para Tesla T4, V100, Bfloat16 para Ampere+
- load_in_4bit: Utiliza menos memória ao reduzir a quantidade de bits de informação. Menos preciso.

In [ ]:
modelo, tokenizador = FastLanguageModel.from_pretrained(
    model_name = checkpoint_modelo,
    max_seq_length=2048,
    dtype = None,
    load_in_4bit=True
)

==((====))==  Unsloth 2024.11.10: Fast Llama patching. Transformers:4.46.2.
   \\   /|    GPU: Tesla T4. Max memory: 14.748 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu121. CUDA: 7.5. CUDA Toolkit: 12.1. Triton: 3.1.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.28.post3. FA2 = False]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


/usr/local/lib/python3.10/dist-packages/transformers/quantizers/auto.py:186: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/230 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.6k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/345 [00:00<?, ?B/s]

In [ ]:
modelo

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096, padding_idx=128004)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaExtendedRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): L

In [ ]:
tokenizador

PreTrainedTokenizerFast(name_or_path='unsloth/meta-llama-3.1-8b-bnb-4bit', vocab_size=128000, model_max_length=131072, is_fast=True, padding_side='left', truncation_side='right', special_tokens={'bos_token': '<|begin_of_text|>', 'eos_token': '<|end_of_text|>', 'pad_token': '<|finetune_right_pad_id|>'}, clean_up_tokenization_spaces=True),  added_tokens_decoder={
	128000: AddedToken("<|begin_of_text|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128001: AddedToken("<|end_of_text|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128002: AddedToken("<|reserved_special_token_0|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128003: AddedToken("<|reserved_special_token_1|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128004: AddedToken("<|finetune_right_pad_id|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	12

### **Gerando consultas com o modelo**

Com o modelo carregado, podemos utilizá-lo para gerar texto a partir de um prompt. Vamos testar o modelo para que ele realize a tarefa de gerar querys SQL a partir de uma pergunta.

In [ ]:
prompt = 'Me dê uma query SQL para saber quantas pessoas tem mais de 56 anos.'

In [ ]:
prompt_tokenizado = tokenizador([prompt], return_tensors='pt').to('cuda')

In [ ]:
prompt_tokenizado

{'input_ids': tensor([[128000,   7979,    294,   5615,  10832,   3319,   8029,   3429,  42104,
          10484,    300,  47062,   1592,  10071,    409,    220,   3487,  38101,
             13]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]],
       device='cuda:0')}

In [ ]:
from transformers import TextStreamer

In [ ]:
FastLanguageModel.for_inference(modelo)
streamer_texto = TextStreamer(tokenizador)

_ = modelo.generate(**prompt_tokenizado, streamer = streamer_texto, max_new_tokens = 128)

<|begin_of_text|>Me dê uma query SQL para saber quantas pessoas tem mais de 56 anos. Por favor, me dê uma query SQL para saber quantas pessoas tem mais de 56 anos. Pessoas com mais de 56 anos. 1. 2. 3. 4. 5. 6. 7. 8. 9. 10. 11. 12. 13. 14. 15. 16. 17. 18. 19. 20. 21. 22. 23. 24. 25. 26. 27. 28. 29. 30. 31. 32. 


O modelo pode gerar um texto inesperado e alterar o objetivo da pergunta inicial, tirando todo o propósito de utilizar o modelo para gerar a query correta.